In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from scripts.query_parser import QueryParser

parser = QueryParser()

In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/listing_sample_cleaned.csv"
)

print("Columns:")
print(df.columns.tolist())

print("\nExample cities:")
print(
    sorted(
        df["L_City"]
        .dropna()
        .unique()
    )[:30]
)

Columns:
['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'remarks', 'cleaned_remarks']

Example cities:
['29 Palms', 'Adelanto', 'Agua Dulce', 'Aguanga', 'Aliso Viejo', 'Altadena', 'Anaheim', 'Anaheim Hills', 'Angelus Oaks', 'Antelope', 'Antioch', 'Anza', 'Apple Valley', 'Aptos', 'Arcadia', 'Arroyo Grande', 'Artesia', 'Atwater', 'Azusa', 'Bakersfield', 'Baldwin Park', 'Banning', 'Barstow', 'Bass Lake', 'Bay Point', 'Beaumont', 'Bellflower', 'Berkeley', 'Beverly Hills', 'Big Bear']


In [4]:
import json
from pathlib import Path

project_root = Path("..").resolve()

schema = {
    "columns": {
        "city": "L_City",
        "bedrooms": "beds",
        "bathrooms": "baths",
        "price": "price",
        "remarks": "cleaned_remarks"
    },
    "valid_cities": sorted(
        df["L_City"]
        .dropna()
        .unique()
        .tolist()
    ),
    "ranges": {
        "price": {
            "min": 100000,
            "max": 10000000
        },
        "bedrooms": {
            "min": 1,
            "max": 10
        },
        "bathrooms": {
            "min": 0,
            "max": 10
        }
    }
}

schema_path = project_root / "data" / "schema.json"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(schema, file, indent=2)

print("Created:", schema_path)
print("Valid cities:", len(schema["valid_cities"]))

Created: /home/cestame/nlp-internship/data/schema.json
Valid cities: 349


In [5]:
df = pd.read_csv(
    "../data/processed/listing_sample_cleaned.csv"
)

print("Columns:")
print(df.columns.tolist())

print("\nExample cities:")
print(
    sorted(
        df["L_City"]
        .dropna()
        .unique()
    )[:30]
)

Columns:
['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'remarks', 'cleaned_remarks']

Example cities:
['29 Palms', 'Adelanto', 'Agua Dulce', 'Aguanga', 'Aliso Viejo', 'Altadena', 'Anaheim', 'Anaheim Hills', 'Angelus Oaks', 'Antelope', 'Antioch', 'Anza', 'Apple Valley', 'Aptos', 'Arcadia', 'Arroyo Grande', 'Artesia', 'Atwater', 'Azusa', 'Bakersfield', 'Baldwin Park', 'Banning', 'Barstow', 'Bass Lake', 'Bay Point', 'Beaumont', 'Bellflower', 'Berkeley', 'Beverly Hills', 'Big Bear']


In [6]:
from scripts.query_parser import QueryParser

parser = QueryParser()

queries = [
    "3 bed in Irvine with community pool under 700k",
    "2 bed in Anaheim without community pool",
    "home with walk-in closet and granite countertops"
]

for query in queries:
    filters = parser.parse(query)
    sql, params = parser.to_sql(filters)

    print("\nQuery:", query)
    print("Filters:", filters)
    print("SQL:", sql)
    print("Parameters:", params)


Query: 3 bed in Irvine with community pool under 700k
Filters: {'price_max': 700000, 'bedrooms': 3, 'city': 'Irvine', 'amenities': ['community pool']}
SQL: SELECT * FROM rets_property WHERE L_SystemPrice <= %s AND L_Keyword2 = %s AND L_City = %s AND L_Remarks LIKE %s
Parameters: [700000, 3, 'Irvine', '%community pool%']

Query: 2 bed in Anaheim without community pool
Filters: {'bedrooms': 2, 'city': 'Anaheim', 'excluded_amenities': ['community pool']}
SQL: SELECT * FROM rets_property WHERE L_Keyword2 = %s AND L_City = %s AND L_Remarks NOT LIKE %s
Parameters: [2, 'Anaheim', '%community pool%']

Query: home with walk-in closet and granite countertops
Filters: {'amenities': ['granite countertops', 'walk-in closet']}
SQL: SELECT * FROM rets_property WHERE L_Remarks LIKE %s AND L_Remarks LIKE %s
Parameters: ['%granite countertops%', '%walk-in closet%']


The first two entries are fine, but the third one revealed a bug. The capture starts after the word in, so it doesn't account for the entire phrase, only the part starting after in. Changed the regex in my query parser to detect "in" only if there is directly a space before it under "city," this would eliminate the mistake, since walk-in is a hyphenated phrase.

In [7]:
###checking if bath and bedroom wording conflicts with price.

In [8]:
from scripts.query_parser import QueryParser

parser = QueryParser()

queries = [
    "at least 3 bedrooms",
    "up to 4 bedrooms",
    "at least 2 bathrooms",
    "up to 3 bathrooms"
]

for query in queries:
    print("\nQuery:", query)
    print("Filters:", parser.parse(query))


Query: at least 3 bedrooms
Filters: {'bedrooms_min': 3}

Query: up to 4 bedrooms
Filters: {'bedrooms_max': 4}

Query: at least 2 bathrooms
Filters: {'bathrooms_min': 2.0}

Query: up to 3 bathrooms
Filters: {'bathrooms_max': 3.0}


In [9]:
from scripts.query_parser import QueryParser

parser = QueryParser()

queries = [
    "at least 3 bedrooms",
    "up to 4 bedrooms",
    "at least 2 bathrooms",
    "up to 3 bathrooms",
    "home with walk-in closet and granite countertops"
]

for query in queries:
    print("\nQuery:", query)
    print("Filters:", parser.parse(query))


Query: at least 3 bedrooms
Filters: {'bedrooms_min': 3}

Query: up to 4 bedrooms
Filters: {'bedrooms_max': 4}

Query: at least 2 bathrooms
Filters: {'bathrooms_min': 2.0}

Query: up to 3 bathrooms
Filters: {'bathrooms_max': 3.0}

Query: home with walk-in closet and granite countertops
Filters: {'amenities': ['granite countertops', 'walk-in closet']}


In [10]:
import importlib
import scripts.query_parser as query_parser

importlib.reload(query_parser)

QueryParser = query_parser.QueryParser
parser = QueryParser()

print(query_parser.__file__)

/home/cestame/nlp-internship/scripts/query_parser.py


In [11]:
from scripts.query_parser import QueryParser

parser = QueryParser()

queries = [
    "at least 3 bedrooms",
    "up to 4 bedrooms",
    "at least 2 bathrooms",
    "up to 3 bathrooms",
    "home with walk-in closet and granite countertops"
]

for query in queries:
    print("\nQuery:", query)
    print("Filters:", parser.parse(query))


Query: at least 3 bedrooms
Filters: {'bedrooms_min': 3}

Query: up to 4 bedrooms
Filters: {'bedrooms_max': 4}

Query: at least 2 bathrooms
Filters: {'bathrooms_min': 2.0}

Query: up to 3 bathrooms
Filters: {'bathrooms_max': 3.0}

Query: home with walk-in closet and granite countertops
Filters: {'amenities': ['granite countertops', 'walk-in closet']}


In [12]:
import sys
import importlib
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import scripts.query_parser as query_parser
importlib.reload(query_parser)

QueryParser = query_parser.QueryParser
parser = QueryParser()

print("Loaded from:", query_parser.__file__)
print(parser.parse("at least 3 bedrooms"))

Loaded from: /home/cestame/nlp-internship/scripts/query_parser.py
{'bedrooms_min': 3}


In [13]:
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_week4.py",
        "-v",
    ],
    cwd=project_root,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.1.1, pluggy-1.6.0 -- /home/cestame/nlp-internship/venv/bin/python
cachedir: .pytest_cache
rootdir: /home/cestame/nlp-internship
plugins: anyio-4.15.1, platformdirs-4.12.2
collecting ... collected 59 items

tests/test_week4.py::test_price_queries[under 700k-price_max-700000] PASSED [  1%]
tests/test_week4.py::test_price_queries[below 500k-price_max-500000] PASSED [  3%]
tests/test_week4.py::test_price_queries[less than 1.2m-price_max-1200000] PASSED [  5%]
tests/test_week4.py::test_price_queries[under $850000-price_max-850000] PASSED [  6%]
tests/test_week4.py::test_price_queries[under $850,000-price_max-850000] PASSED [  8%]
tests/test_week4.py::test_price_queries[up to $900000-price_max-900000] PASSED [ 10%]
tests/test_week4.py::test_price_queries[at most 750k-price_max-750000] PASSED [ 11%]
tests/test_week4.py::test_price_queries[max $600000-price_max-600000] PAS

In [14]:
df = pd.read_csv(
    "../data/processed/sample_queries.csv"
)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nFirst five rows:")
print(df.head().to_string(index=False))

Rows: 50
Columns: ['query_id', 'query', 'intent']

First five rows:
 query_id                              query   intent
        1         show me homes in San Diego browsing
        2              find condos in Irvine browsing
        3 show me houses with swimming pools browsing
        4     find homes with mountain views browsing
        5  show me townhouses near the beach browsing


In [15]:
df = pd.read_csv(
    "../data/processed/sample_queries.csv"
)

for _, row in df.iterrows():
    print(
        f"{row['query_id']:>2}. "
        f"[{row['intent']}] "
        f"{row['query']}"
    )

 1. [browsing] show me homes in San Diego
 2. [browsing] find condos in Irvine
 3. [browsing] show me houses with swimming pools
 4. [browsing] find homes with mountain views
 5. [browsing] show me townhouses near the beach
 6. [browsing] find properties with large backyards
 7. [browsing] show me homes in gated communities
 8. [browsing] find houses with home offices
 9. [browsing] show me homes with open floor plans
10. [browsing] find condos with community fitness centers
11. [browsing] show me single-story homes
12. [browsing] find homes with outdoor kitchens
13. [browsing] show me properties near shopping
14. [browsing] find houses with hardwood floors
15. [browsing] show me homes with three-car garages
16. [browsing] find properties with ocean views
17. [browsing] show me homes with updated kitchens
18. [researching] which Irvine neighborhoods have the lowest HOA fees
19. [researching] compare condos and townhouses in San Diego
20. [researching] which areas have highly rated scho

In [16]:
queries_df = pd.read_csv(
    project_root
    / "data"
    / "processed"
    / "sample_queries.csv"
)

results = []

for query in queries_df["query"]:
    filters = parser.parse(query)

    results.append({
        "query": query,
        "filters": filters,
        "parsed": len(filters) > 0,
    })

results_df = pd.DataFrame(results)

parsed_count = results_df["parsed"].sum()
total_count = len(results_df)
coverage = parsed_count / total_count

print("Queries parsed:", parsed_count)
print("Total queries:", total_count)
print(f"Current parser coverage: {coverage:.2%}")

Queries parsed: 49
Total queries: 50
Current parser coverage: 98.00%


##current parser coverage percentage: 44.00%
The biggest missing patterns are:
- Written numbers: three-bedroom, two bathrooms
- Property types: condo, townhouse, single-story home
- Plural/synonym amenities: swimming pools, mountain views
- Multiple values: Irvine and San Diego
- Negation: without solar, no HOA
- Availability: this weekend, tomorrow
- Comparison/research queries

In [17]:
queries_df = pd.read_csv(
    project_root
    / "data"
    / "processed"
    / "sample_queries.csv"
)

results = []

for query in queries_df["query"]:
    filters = parser.parse(query)

    results.append({
        "query": query,
        "filters": filters,
        "parsed": len(filters) > 0,
    })

results_df = pd.DataFrame(results)

parsed_count = results_df["parsed"].sum()
total_count = len(results_df)
coverage = parsed_count / total_count

print("Queries parsed:", parsed_count)
print("Total queries:", total_count)
print(f"Current parser coverage: {coverage:.2%}")

Queries parsed: 49
Total queries: 50
Current parser coverage: 98.00%


In [18]:
number_word_pattern = (
    r"\b(?:one|two|three|four|five|six|seven|eight|"
    r"nine|ten)[-\s]+(?:bed|bedroom|bath|bathroom)"
)

number_word_results = results_df[
    results_df["query"].str.contains(
        number_word_pattern,
        case=False,
        regex=True,
    )
]

number_word_results[["query", "filters", "parsed"]]

,query,filters,parsed
30,what is the average price of a three-bedroom home,"{'bedrooms': 3, 'operation': 'average'}",True
34,find a three-bedroom home under 800000 in Irvine,"{'price_max': 800000, 'bedrooms': 3, 'city': '...",True
36,find a two-bedroom condo under 600000 near dow...,"{'price_max': 600000, 'bedrooms': 2, 'property...",True
40,find a four-bedroom home with a private pool i...,"{'bedrooms': 4, 'city': 'Irvine', 'amenities':...",True
44,find a three-bedroom townhouse with no HOA,"{'bedrooms': 3, 'property_types': ['townhouse'...",True
46,find a condo with two bathrooms and assigned p...,"{'bathrooms': 2.0, 'property_types': ['condo']...",True


In [19]:
number_word_parsed = number_word_results["parsed"].sum()
number_word_total = len(number_word_results)

print("Number-word queries parsed:", number_word_parsed)
print("Number-word query total:", number_word_total)

if number_word_total > 0:
    percentage = number_word_parsed / number_word_total
    print(f"Number-word coverage: {percentage:.2%}")

Number-word queries parsed: 6
Number-word query total: 6
Number-word coverage: 100.00%


In [20]:
results_df[
    results_df["parsed"] == False
][["query"]]

,query
27,which areas have lower property taxes
